<div dir="rtl" align="right">

# ضبطُ المُعاملاتِ الفائقةِ بِالبحثِ الشبكيِّ

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُجري بحثاً شبكيّاً على مُعاملاتِ C و penalty و solver لِلانحدارِ اللوجستيِّ بِاستخدامِ التحقّقِ المتقاطعِ بِـ 5 طيّاتٍ.

## ماذا يَعمَلُ هذا الدفترُ؟

يَحسبُ سماتِ قُوّةِ النطاقِ، ويُقيّسُها، ويُشغّلُ GridSearchCV على شبكةِ المُعاملاتِ لِإيجادِ أفضلِ تكوينٍ.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ شريطيٌّ لِمتوسطِ دقّةِ التحقّقِ المتقاطعِ لِكلِّ قيمةِ C
- خريطةٌ حراريّةٌ لِنتائجِ التحقّقِ المتقاطعِ لِمُركّباتِ C × Penalty
- أفضلُ المُعاملاتِ وأفضلُ دقّةِ تحقّقٍ متقاطعٍ مَطبوعةٌ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| cv | 5 |
| C | 0.01, 0.1, 1, 10, 100 |
| penalty | l1, l2 |

</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تشغيلُ البحثِ الشبكيِّ

</div>


In [ ]:
from scipy.signal import welch

FS = 250
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

print(f'Feature matrix shape: {features.shape}')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
}

clf = LogisticRegression(max_iter=1000, random_state=42)
grid = GridSearchCV(clf, param_grid, cv=5, scoring='accuracy')
grid.fit(features_scaled, labels)

print(f'Best parameters: {grid.best_params_}')
print(f'Best CV accuracy: {grid.best_score_:.4f}')

results = grid.cv_results_
C_values = [0.01, 0.1, 1, 10, 100]
penalties = ['l1', 'l2']

mean_acc_per_C = np.zeros(len(C_values))
for i, c in enumerate(C_values):
    mask_c = results['param_C'] == c
    mean_acc_per_C[i] = np.mean(results['mean_test_score'][mask_c])

pivot = np.zeros((len(penalties), len(C_values)))
for i, p in enumerate(penalties):
    for j, c in enumerate(C_values):
        mask_p = (results['param_penalty'] == p) & (results['param_C'] == c)
        pivot[i, j] = results['mean_test_score'][mask_p][0]


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- المخططُ الشريطيُّ يُظهرُ أيُّ قيمةِ C تُعطي أفضلَ دقّةٍ مُتوسّطةٍ
- الخريطةُ الحراريّةُ تَكشفُ التفاعلاتِ بينَ C ونوعِ penalty
- penalty من نوع L1 قد يَعملُ بشكلٍ مُختلفٍ عن L2 حسبَ C

</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pivot_text = [[f'{pivot[i, j]:.3f}' for j in range(len(C_values))] for i in range(len(penalties))]

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Mean CV Accuracy per C (averaged across penalties)',
    'CV Results (C x Penalty)'))

fig.add_trace(go.Bar(x=[str(c) for c in C_values], y=mean_acc_per_C, marker_color='steelblue', name='Accuracy'), row=1, col=1)
fig.add_trace(go.Heatmap(z=pivot, x=[str(c) for c in C_values], y=penalties, colorscale='YlGnBu',
    text=pivot_text, texttemplate='%{text}', textfont={'size': 14}, name='CV', showscale=True), row=2, col=1)

fig.update_xaxes(title_text='C value', row=1, col=1)
fig.update_yaxes(title_text='Mean CV Accuracy', row=1, col=1)
fig.update_xaxes(title_text='C', row=2, col=1)
fig.update_yaxes(title_text='Penalty', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='Grid Search Hyperparameter Tuning')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- البحثُ الشبكيُّ يَستكشفُ فضاءَ المُعاملاتِ بِشكلٍ منهجيٍّ لِإيجادِ أفضلِ تكوينٍ
- التحقّقُ المتقاطعُ يُوفّرُ تقديراً مُتيناً لأداءِ التعميمِ
- مُعاملُ C يَتحكّمُ في قُوّةِ التنظيمِ: C صغيرٌ = تنظيمٌ قويٌّ

</div>
